In [1]:
import re
import string
import pandas as pd
from datasets import load_dataset
import os

In [2]:
# optional - do not use if you don't have the dataset in cache
os.environ["HF_DATASETS_OFFLINE"] = "1"

In [3]:
# filter for English language and US region rows
ds = load_dataset("allenai/WildChat-1M", num_proc=4)
filtered = ds["train"].filter(
    lambda x: x["language"] == "English" and x["country"] == "United States" and x["state"] is not None and x["state"] != ""
)
df = pd.DataFrame(filtered)
print(f"Rows after filter: {len(df):,}")

Filter:   0%|          | 0/837989 [00:00<?, ? examples/s]

Rows after filter: 128,943


In [4]:
# save intermediate df
df.to_parquet("wildchat_en_us.parquet", index=False)

In [6]:
def extract_user_text(conversation) -> str:
    """
    Given a list of turn dicts, keep only the first role=='user' turn and pull the content.
    """
    for turn in conversation:
        if turn.get("role") == "user" and isinstance(turn.get("content"), str):
            return turn["content"]
    return ""

df["user_text"] = df["conversation"].apply(extract_user_text)

In [7]:
# Deduplicate user prompts within the same hashed_ip
from datasketch import MinHash, MinHashLSH

df = df[df["user_text"].str.strip().ne("")].reset_index(drop=True)

def get_shingles(text: str, n: int = 3) -> set[bytes]:
    text = text.lower()
    return {text[i:i+n].encode("utf-8") for i in range(len(text) - n + 1)}

def make_minhash(text: str, num_perm: int = 128) -> MinHash:
    m = MinHash(num_perm=num_perm)
    for shingle in get_shingles(text):
        m.update(shingle)
    return m

THRESHOLD = 0.85
NUM_PERM = 128

def deduplicate_group(group_df: pd.DataFrame) -> pd.DataFrame:
    """Run MinHash LSH deduplication within a single hashed_ip group."""
    group_df = group_df.reset_index(drop=True)
    lsh = MinHashLSH(threshold=THRESHOLD, num_perm=NUM_PERM)
    cluster_map = {}
    clusters = {}

    for idx, row in group_df.iterrows():
        m = make_minhash(row["user_text"], num_perm=NUM_PERM)
        results = lsh.query(m)
        if results:
            cluster_id = cluster_map[int(results[0])]
            cluster_map[idx] = cluster_id
            clusters[cluster_id].append(idx)
        else:
            cluster_map[idx] = idx
            clusters[idx] = [idx]
        lsh.insert(str(idx), m)

    keep_indices = {
        max(members, key=lambda i: len(group_df.at[i, "user_text"]))
        for members in clusters.values()
    }
    return group_df[group_df.index.isin(keep_indices)]

df_deduped = (
    df.groupby("hashed_ip", group_keys=False)
    .apply(deduplicate_group)
    .reset_index(drop=True)
)

n_removed = len(df) - len(df_deduped)
print(f"Removed {n_removed:,} near-duplicate rows ({n_removed/len(df):.1%})")
print(f"Rows remaining: {len(df_deduped):,}")

Removed 58,884 near-duplicate rows (45.7%)
Rows remaining: 69,842


In [8]:
# save intermediate df
df_deduped.to_parquet("wildchat_dedup.parquet", index=False)

In [17]:
_punct_table = str.maketrans("", "", string.punctuation)

def clean_text(text: str) -> str:
    """Lowercase and strip all punctuation from text."""
    return text.lower().translate(_punct_table)

df_final = df_deduped.copy()
df_final["user_text"] = df_deduped["user_text"].apply(clean_text)

# Remove rows where user_text is empty after punctuation removal
rows_before = len(df_final)
df_final = df_final[df_final["user_text"].str.strip().astype(bool)]
print(f"Dropped {rows_before - len(df_final)} rows with empty user_text after cleaning.")

KEEP_COLS = [
    "conversation_hash",
    "model",
    "timestamp",
    "hashed_ip",
    "user_text",
    "state"
]

df_final = df_final.reindex(columns=KEEP_COLS)

print(f"\nFinal dataset shape : {df_final.shape}")
print("\nSample rows:")
print(df_final.head(3).to_string(index=False))

df_final.to_parquet("wildchat_log_odds.parquet", index=False)

Dropped 1 rows with empty user_text after cleaning.

Final dataset shape : (69841, 6)

Sample rows:
               conversation_hash              model                 timestamp                                                        hashed_ip                                                                                            user_text      state
bfb162e7842ea1a50992342909397eae gpt-3.5-turbo-0613 2023-09-30 07:49:00+00:00 000bf6f53c40286dabf2c9799d261b32be46be3b079ea953d3d488bb5a2d7af7                                                                                                   hi California
c36175e7d280fe0eb8fb699084512273 gpt-4-1106-preview 2023-11-30 13:48:58+00:00 000e1503dbe12557a1a7e5f60d06e9cb4adafdb88da042da1ac0565b035c2088 write me a narrative roleplay of beast boy vs bane have beast boy transform into lesserknown animals       Ohio
330b792806a08893ebb7f5501c161b5c gpt-3.5-turbo-0613 2023-12-06 01:43:09+00:00 000e5b67571f779e148433dec2aaa9d99de6dfc91784d88a537e042d1